# Design: code_* overlay snapshot cache with Git fsmonitor

**Status:** Approved direction; written specification awaiting final user review  
**Date:** 2026-08-24  
**Design epic:** `bd-3h31`  
**Scope:** A benchmark probe and the production design it must validate. No production behavior changes are authorized by this document alone.

This specification replaces TTL-based freshness with a Git-owned change signal, a per-worktree `path → OID/state` snapshot, and an exact-scan fallback. It also removes the duplicate base query from the target request journey.

## 1. Approved decision

Use Git's built-in `fsmonitor--daemon` as the preferred change detector, but integrate it through a redesigned steady-state command path:

1. Require the probe release gate or an explicit feature enablement before production routing can select fsmonitor.
2. Capability-check the Git build and worktree filesystem once per worktree.
3. Run one warm `git status --porcelain=v2 -z --untracked-files=all` with process-scoped `core.fsmonitor=true` and `core.untrackedCache=true`.
4. Derive changed paths from that response and batch-hash only worktree files whose current OID is not reported by Git.
5. Reuse an overlay snapshot while its normalized Git state fingerprint, HEAD, index identity, and indexed graph identity remain unchanged.
6. Use the existing exact scan when the release gate is disabled, or on unsupported Git versions, daemon failure, remote/unsupported filesystems, inconsistent observations, or repeated races.

Git owns the opaque fsmonitor token. SPUR does not parse or depend on Git's private daemon IPC protocol, and it does not permanently modify user or global Git configuration.

A configuration-only change around the current `status + ls-files -t + ls-files -s` bundle is explicitly rejected: it was slower in the grounding probe and retains the full tracked-file scans.

## 2. Measured problem and latency budget

The observed hot `code_*` request is approximately 142 ms:

| Segment | Observed cost |
|---|---:|
| Primary Parquet query | ~40 ms |
| Overlay change discovery | 53.8 ms |
| Repeated query through the overlay path | ~30–40 ms |
| Metadata, routing, and serialization remainder | ~8–18 ms |

The release-mode dirty-set benchmark covered 3,061 indexed files and 170 changed paths. A separate isolated Git probe measured:

| Command shape | Mean |
|---|---:|
| Plain `git status` | 23.8 ms |
| Fsmonitor-backed `git status` | 26.4 ms |
| Existing three-command bundle, plain | 34.7 ms |
| Existing bundle with fsmonitor on status | 47.7 ms |
| Persisted fsmonitor + untracked-cache bundle | 64.7 ms |

These measurements mean fsmonitor is a compatibility and change-locality mechanism, not a free speedup. The probe must demonstrate that removing the two full `ls-files` sweeps and hashing only changed paths overcomes daemon IPC and cache-maintenance overhead.

Relevant implementation seams:

- `crates/spur-graph/src/mcp/mod.rs::code_search_response`
- `crates/spur-graph/src/mcp/mod.rs::overlay_response_for_backend`
- `crates/spur-graph/src/mcp/mod.rs::changed_paths_for_overlay`
- `crates/spur-graph/src/mcp/mod.rs::current_file_oids_via_git`
- `crates/spur-graph/src/query_client.rs::ParquetClient::file_oids`

## 3. Goals and non-goals

### Goals

- Preserve overlay correctness for committed graph lag, staged, unstaged, untracked, deleted, renamed, sparse-checkout, HEAD/index, and linked-worktree changes.
- Make steady-state discovery proportional to changed paths rather than all tracked paths.
- Keep a per-worktree snapshot valid without a correctness TTL while Git reports the same state.
- Ensure a request executes the base Parquet query at most once.
- Fall back conservatively without returning stale overlay data.
- Compare behavior and latency across three project-size classes.

### Non-goals

- Calling Git's private fsmonitor IPC protocol directly.
- Replacing Git with an in-process filesystem watcher.
- Permanently changing repository, global, or system Git configuration.
- Guaranteeing fsmonitor operation on network-mounted or otherwise unsupported filesystems.
- Changing the public `code_*` request or response schema.
- Treating the benchmark probe as production authorization.

## 4. Runtime architecture and data flow

### Steady-state request

1. Resolve the canonical linked-worktree identity and indexed graph identity.
2. Read current HEAD and an index identity stamp.
3. Ask Git status for the fsmonitor-filtered staged, unstaged, untracked, deleted, and rename set.
4. Normalize paths and filter to supported source files.
5. Reuse Git-reported index OIDs; batch `git hash-object --stdin-paths` only for changed worktree files requiring current content OIDs.
6. Represent deletions as tombstones and renames as an old-path tombstone plus new-path OID.
7. Build a stable fingerprint from sorted `(path, state, OID)` entries plus HEAD, index identity, and indexed graph identity.
8. Reuse or singleflight-build the overlay delta keyed by that fingerprint.
9. Apply the delta to the already-produced base query result; do not execute the Parquet query again.

### Indexed graph behind HEAD

A clean `git status` does not detect commits made after the graph was indexed. When `current_head != indexed_head`, execute a conditional `git diff --name-status -z indexed_head current_head` and obtain current commit OIDs for those paths. This is outside the one-status steady state and is required for correctness.

### Consistency window

Capture HEAD/index/status, hash changed files with the existing stat-aware OID cache, and then revalidate HEAD/index plus per-file metadata. Retry once on a race. A second inconsistent observation or any Git failure routes the request to the exact scanner.

In [ ]:
flowchart TD
    SPEC["`@spec OVERLAY-ROUTING
@type Route = enum[fsmonitor_native, exact_scan]
@input release_enabled: Bool
@input built_in_supported: Bool
@input local_filesystem: Bool
@input watcher_healthy: Bool
@output status: Route
@requires INPUTS: true`"]
    FSM["`@branch FSMONITOR
@when release_enabled = true and built_in_supported = true and local_filesystem = true and watcher_healthy = true
@ensures FSMONITOR_ROUTE: status = fsmonitor_native`"]
    FALLBACK["`@branch EXACT_FALLBACK
@when release_enabled = false or built_in_supported = false or local_filesystem = false or watcher_healthy = false
@ensures FALLBACK_ROUTE: status = exact_scan`"]
    CHECK["`@verify ROUTE_DETERMINISTIC: prove determinism
@verify ROUTE_COVERED: prove partition_coverage
@verify ROUTE_EXCLUSIVE: prove partition_exclusive
@verify BOTH_ROUTES: witness each status`"]
    SPEC --> FSM --> CHECK
    SPEC --> FALLBACK --> CHECK

## 5. Snapshot identity and cache ownership

The cache is per canonical worktree, not per repository common directory and not per OID alone.

```text
OverlaySnapshotKey
  worktree_identity
  indexed_graph_content_hash
  indexed_head_oid
  current_head_oid
  index_identity
  normalized_changed_set_fingerprint

OverlaySnapshot
  path_state: path -> Tracked(oid) | Untracked(oid) | Deleted
  overlay_delta
  source: FsmonitorNative | ExactFallback
  observation_health
```

OID-only identity is insufficient: the same blob can appear at several paths, a deletion has no current OID, and rename semantics involve both path identities. The normalized path/state/OID set is therefore the content key; Git's internal fsmonitor token is the efficient validation mechanism.

The existing per-file OID cache remains useful for batch hashing and race detection. The existing overlay singleflight/cache remains the owner of extraction reuse, with its key upgraded to the complete snapshot identity.

## 6. Safety and invalidation contract

A cached overlay may be served only when its snapshot is fresh for the current worktree and graph identity.

- A successful exact or fsmonitor-native observation can move the snapshot to **Valid**.
- A staged, unstaged, untracked, deletion, rename, HEAD, or index change invalidates the prior fingerprint.
- Daemon restart, event loss, unsupported capability, command failure, malformed output, or a repeated consistency race disables serving and triggers exact scanning.
- The fallback result may repopulate the snapshot, tagged as `ExactFallback`.
- False-positive changed paths are acceptable; false negatives are not.
- Cache lifetime is change-based. No TTL is required for correctness.
- Optional operational expiry may reclaim memory, but eviction must not be described as freshness validation.

In [ ]:
stateDiagram-v2
    [*] --> Unknown
    Unknown --> Scanning: request
    Scanning --> Valid: scan_ok
    Valid --> Valid: cache_hit
    Valid --> Invalidated: change
    Valid --> Scanning: event_loss
    Invalidated --> Scanning: rescan

    note right of Unknown
      @spec OVERLAY-SNAPSHOT-LIFECYCLE
      @type CacheEvent = enum[request, scan_ok, cache_hit, change, event_loss, rescan]
      @input event: CacheEvent
      @state-var serving_allowed: Bool
      @state-var snapshot_fresh: Bool
      @requires INIT_SERVING: serving_allowed = false
      @requires INIT_FRESH: snapshot_fresh = false
      @state Unknown
      @invariant NO_STALE_SERVE: not serving_allowed or snapshot_fresh
    end note

    note right of Scanning
      @state Scanning
      @transition REQUEST_SCAN
      @from Unknown
      @to Scanning
      @event event = request
      @guard serving_allowed = false
      @update serving_allowed' = false
      @update snapshot_fresh' = false
    end note

    note right of Valid
      @state Valid
      @transition SCAN_OK
      @from Scanning
      @to Valid
      @event event = scan_ok
      @guard serving_allowed = false
      @update serving_allowed' = true
      @update snapshot_fresh' = true
    end note

    note right of Valid
      @transition CACHE_HIT
      @from Valid
      @to Valid
      @event event = cache_hit
      @guard serving_allowed = true and snapshot_fresh = true
      @update serving_allowed' = true
      @update snapshot_fresh' = true
    end note

    note right of Invalidated
      @state Invalidated
      @transition INVALIDATE
      @from Valid
      @to Invalidated
      @event event = change
      @guard serving_allowed = true
      @update serving_allowed' = false
      @update snapshot_fresh' = false
    end note

    note right of Scanning
      @transition EVENT_LOSS
      @from Valid
      @to Scanning
      @event event = event_loss
      @guard serving_allowed = true
      @update serving_allowed' = false
      @update snapshot_fresh' = false
    end note

    note right of Scanning
      @transition RESCAN
      @from Invalidated
      @to Scanning
      @event event = rescan
      @guard serving_allowed = false
      @update serving_allowed' = false
      @update snapshot_fresh' = false
      @verify INIT_SAFE: prove initiate NO_STALE_SERVE
      @verify PRESERVE_REQUEST: prove preserve NO_STALE_SERVE on REQUEST_SCAN
      @verify PRESERVE_SCAN_OK: prove preserve NO_STALE_SERVE on SCAN_OK
      @verify PRESERVE_HIT: prove preserve NO_STALE_SERVE on CACHE_HIT
      @verify PRESERVE_INVALIDATE: prove preserve NO_STALE_SERVE on INVALIDATE
      @verify PRESERVE_LOSS: prove preserve NO_STALE_SERVE on EVENT_LOSS
      @verify PRESERVE_RESCAN: prove preserve NO_STALE_SERVE on RESCAN
    end note

## 7. Probe methodology

The probe is release-mode and isolated from user Git configuration. Filesystem and Git-index mutations used to create dirty states or warm Git caches occur only in disposable clones/worktrees; tracked source in the user's worktree is not modified. The probe must measure these variants under the same fixtures:

| Variant | Purpose |
|---|---|
| Current exact path | Baseline |
| Current command bundle + fsmonitor | Negative control; proves a flag alone is insufficient |
| Fsmonitor-native status + changed-only OIDs | Preferred candidate |
| Forced exact fallback | Correctness and degradation baseline |

### Workload matrix

- Three project classes: small, medium, and large by tracked source-file count.
- Clean, one changed file, approximately 170 changed paths, and high-dirty-set cases.
- Tracked edit, staged edit, untracked create, delete, rename, HEAD advance, index replacement, sparse checkout, and linked worktree.
- Cold daemon, warming runs, warm steady state, daemon restart, unsupported filesystem simulation, malformed output, and forced command failure.
- Both optional-index-lock modes are measured because disabling optional locks may reduce contention while also reducing Git cache persistence.

### Recorded metrics

- End-to-end `code_*` p50 and p95.
- Primary Parquet query count and duration.
- Warm snapshot validation, overlay discovery, changed-only OID hashing, delta build, and metadata durations.
- Git subprocess count, wall/user/sys time, and fallback rate.
- Dirty-path count, tracked-source count, and cache-hit classification.
- Output equivalence against the current exact scanner.

The candidate and exact scanner run against identical snapshots. Any path/OID/state mismatch is a correctness failure, regardless of latency.

## 8. Release gate

The conservative latency model uses the observed 54 ms overlay scan and the measured lower bound of 30 ms for the repeated query. On the first request after a change:

```text
current overhead   = 54 + 30 = 84 ms
candidate overhead = 54 conservative recovery + warm snapshot validation
```

The greatest integer warm-validation cost that strictly improves this conservative case is 29 ms. Therefore the **warm snapshot validation/cache-lookup p95** gate is less than 30 ms, not less-than-or-equal. This is not a claim that every changed request completes its entire overlay rebuild below 30 ms.

A release candidate must satisfy all of these conditions:

1. Warm snapshot validation/cache-lookup p95 is below 30 ms.
2. Exactly one base Parquet query executes per request.
3. No correctness mismatch occurs across the full invalidation matrix.
4. All three project-size classes pass.
5. Unsupported or unhealthy fsmonitor conditions use the exact fallback before serving.
6. End-to-end and changed-request improvement is repeatable under the same release-mode workload.

The formal `latency_matrix_passed` input represents condition 6 and is true only when both improvements reproduce across the declared workload matrix.

A faster result that violates any correctness condition is rejected. A correct result that does not beat both the formal warm-validation gate and the measured end-to-end baseline remains a probe result and does not replace the current path.

In [ ]:
flowchart TD
    SPEC["`@spec OVERLAY-PERFORMANCE-GATE
@type Decision = enum[release_candidate, reject]
@input warm_validation_p95_ms: Int
@input base_query_count: Int
@input correctness_failures: Int
@input project_classes_passed: Int
@input latency_matrix_passed: Bool
@output status: Decision
@requires LATENCY_RANGE: warm_validation_p95_ms >= 0 and warm_validation_p95_ms <= 1000
@requires QUERY_RANGE: base_query_count >= 0 and base_query_count <= 4
@requires FAILURE_RANGE: correctness_failures >= 0 and correctness_failures <= 100
@requires PROJECT_RANGE: project_classes_passed >= 0 and project_classes_passed <= 3`"]
    PASS["`@branch PASS
@when warm_validation_p95_ms < 30 and base_query_count = 1 and correctness_failures = 0 and project_classes_passed = 3 and latency_matrix_passed = true
@ensures PASS_DECISION: status = release_candidate`"]
    REJECT["`@branch REJECT
@when warm_validation_p95_ms >= 30 or base_query_count != 1 or correctness_failures != 0 or project_classes_passed != 3 or latency_matrix_passed = false
@ensures REJECT_DECISION: status = reject`"]
    CHECK["`@verify GATE_DETERMINISTIC: prove determinism
@verify GATE_COVERED: prove partition_coverage
@verify GATE_EXCLUSIVE: prove partition_exclusive
@verify BOTH_DECISIONS: witness each status`"]
    SPEC --> PASS --> CHECK
    SPEC --> REJECT --> CHECK

## 9. Failure handling and compatibility

| Condition | Required behavior |
|---|---|
| Git lacks usable built-in fsmonitor | Exact scan |
| Daemon refuses the filesystem | Exact scan |
| Network-mounted or experimental filesystem | Exact scan unless an explicit future policy changes this |
| Linux inotify capacity failure | Exact scan and diagnostic |
| Daemon restart or unhealthy response | Do not serve prior snapshot; exact scan |
| Status parse error or unsupported record | Exact scan |
| HEAD/index changes during observation | Retry once, then exact scan |
| File metadata changes during hashing | Retry once, then exact scan |
| Git command timeout | Terminate child, exact scan, record fallback metric |
| Submodule-only event | Filter unsupported paths; false-positive work is acceptable |
| Concurrent requests | Singleflight one observation/delta build per complete snapshot key |

Capability detection is cached per worktree process lifetime, but a failed request immediately degrades to exact scanning. A later bounded health check may restore fsmonitor eligibility; restoration never reuses a snapshot created before the failure.

Primary references:

- https://git-scm.com/docs/git-fsmonitor--daemon
- https://git-scm.com/docs/git-status
- https://git-scm.com/docs/git-config.html
- https://git-scm.com/docs/githooks

## 10. Task boundaries for the later implementation plan

The implementation plan must preserve this dependency order:

1. **Probe harness and equivalence oracle**  
   Own benchmark fixtures, current-path instrumentation, cross-project workload matrix, and exact-output comparison.

2. **Fsmonitor-native observation adapter**  
   Own capability detection, porcelain-v2 parsing, HEAD-lag diffing, changed-only OID batching, consistency retry, and fallback classification. It depends on the harness contract.

3. **Snapshot cache and query-path integration**  
   Own the complete cache key, singleflight behavior, applying the delta to the first query result, and removing duplicate query execution. It depends on accepted adapter evidence.

4. **Regression and release verification**  
   Own invalidation tests, unsupported-platform fixtures, release-mode benchmarks, and formal-gate evidence. It depends on integration.

Parser/adapter fixtures can be developed independently from benchmark dataset preparation after the harness interfaces are fixed. Cache integration must not begin merely because the adapter is correct; the adapter must also meet the performance gate.

## 11. Risks, evidence limits, and review checklist

### Principal risks

- Git fsmonitor availability varies by Git version, build, platform, and filesystem.
- Warm-cache results can hide daemon startup and index-cache costs.
- `git status` reports changed paths but does not always provide the current worktree blob OID.
- Background status may contend on the Git index; optional-lock behavior must be measured.
- A graph indexed behind HEAD requires an explicit commit delta even when status is clean.
- An apparently stable dirty-path set can still contain files changing during hashing.
- Performance gains on large clean repositories may not reproduce on small or highly dirty repositories.

### Evidence limits

- The existing fsmonitor timings are command microbenchmarks, not evidence that the redesigned path is fast.
- Formal routing and release cells prove finite decision partitions only.
- The lifecycle cell proves the authored invariant initiates and is preserved by every declared transition; it does not prove omitted implementation transitions.
- Production rollout requires probe evidence and a separate approved implementation plan.

### Written-spec review checklist

- Preferred and fallback routes are unambiguous.
- Cache identity includes worktree, graph, Git state, paths, states, and OIDs.
- Every invalidation class has a test fixture.
- The exact scanner remains the correctness oracle and fallback.
- Formal cells execute with fresh hashes and their expected SAT/UNSAT results.
- No production code is changed before this specification is reviewed and the implementation plan is approved.